## Project 

In [1]:
# Advanced Pedestrian Detection System using YOLOv11
# Final Year Project - Computer Vision & Deep Learning

# Dataset: CityPersons (Kaggle)
# Model: YOLOv11 with BiFPN Architecture
# Task: Real-time Pedestrian Detection for Autonomous Systems

In [2]:
# !pip install -U ultralytics opencv-python matplotlib seaborn

In [3]:
# Imports
from ultralytics import YOLO
import torch
import os
import glob
import cv2
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

print("Torch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

Torch version: 2.9.0
CUDA available: False


In [ ]:
DATASET_PATH = "Citypersons"

assert os.path.exists(DATASET_PATH), "Dataset path not found"

In [5]:
dataset_yaml = f"""
path: {DATASET_PATH}
train: images/train
val: images/val
test: images/test

nc: 1
names:
  0: person
"""

with open("citypersons.yaml", "w") as f:
    f.write(dataset_yaml)

print("citypersons.yaml created")

citypersons.yaml created


In [6]:
def clean_labels(label_dir):
    removed = 0
    for root, _, files in os.walk(label_dir):
        for f in files:
            if f.endswith(".txt"):
                path = os.path.join(root, f)
                with open(path) as file:
                    lines = file.readlines()

                valid = []
                for l in lines:
                    cls, x, y, w, h = map(float, l.split())
                    if w > 0 and h > 0:
                        valid.append(l)

                if len(valid) == 0:
                    os.remove(path)
                    removed += 1
                else:
                    with open(path, "w") as file:
                        file.writelines(valid)

    return removed

removed_train = clean_labels(f"{DATASET_PATH}/labels/train")
removed_val = clean_labels(f"{DATASET_PATH}/labels/val")

print("Removed empty labels (train):", removed_train)
print("Removed empty labels (val):", removed_val)


Removed empty labels (train): 0
Removed empty labels (val): 0


In [7]:
def keep_person_only(label_dir):
    for root, _, files in os.walk(label_dir):
        for f in files:
            if f.endswith(".txt"):
                path = os.path.join(root, f)
                with open(path) as file:
                    lines = file.readlines()

                person_only = [l for l in lines if l.startswith("0")]

                with open(path, "w") as file:
                    file.writelines(person_only)

keep_person_only(f"{DATASET_PATH}/labels/train")
keep_person_only(f"{DATASET_PATH}/labels/val")

print("Filtered labels to pedestrian-only")


Filtered labels to pedestrian-only


In [8]:
train_imgs = glob.glob(f"{DATASET_PATH}/images/train/**/*.png", recursive=True)
val_imgs = glob.glob(f"{DATASET_PATH}/images/val/**/*.png", recursive=True)

print("Training images:", len(train_imgs))
print("Validation images:", len(val_imgs))


Training images: 2975
Validation images: 502


In [9]:
model = YOLO("yolov11_bifpn.yaml")
model.info()

WARNING ⚠️ no model scale passed. Assuming scale='n'.
YOLOv11_bifpn summary: 319 layers, 2,590,035 parameters, 2,590,019 gradients, 6.4 GFLOPs


(319, 2590035, 2590019, 6.4406016)

In [ ]:
model.train(
    data="citypersons.yaml",
    epochs=100,
    imgsz=640,
    batch=8,
    device="cpu",          # or "mps"
    hsv_h=0.015,
    hsv_s=0.7,
    hsv_v=0.4,
    degrees=0.0,
    translate=0.1,
    scale=0.5,
    shear=0.0,
    fliplr=0.5,
    flipud=0.0,
    mosaic=1.0,
    mixup=0.0,
    copy_paste=0.0,
    close_mosaic=10,
    project="runs_citypersons",
    name="yolov11_bifpn_pedestrian",
    patience=10,
    pretrained=False
)

New https://pypi.org/project/ultralytics/8.4.3 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.3.24 🚀 Python-3.12.7 torch-2.9.0 CPU (Apple M1)
engine/trainer: task=detect, mode=train, model=yolov11_bifpn.yaml, data=citypersons.yaml, epochs=2, time=None, patience=10, batch=8, imgsz=640, save=True, save_period=-1, cache=False, device=cpu, workers=8, project=runs_citypersons, name=yolov11_bifpn_pedestrian, exist_ok=False, pretrained=False, optimizer=auto, verbose=True, seed=0, deterministic=True, single_cls=False, rect=False, cos_lr=False, close_mosaic=10, resume=False, amp=True, fraction=1.0, profile=False, freeze=None, multi_scale=False, overlap_mask=True, mask_ratio=4, dropout=0.0, val=True, split=val, save_json=False, save_hybrid=False, conf=None, iou=0.7, max_det=300, half=False, dnn=False, plots=True, source=None, vid_stride=1, stream_buffer=False, visualize=False, augment=False, agnostic_nms=False, classes=None, retina_masks=False, embed=None, show=False, save_fr

train: Scanning /Users/hariraj/Desktop/STATBRIO/S_pdetection/Citypersons/labels/train/aachen.cache... 2500 images, 475 backgrounds, 0 corrupt: 100%|██████████| 2975/2975 [00:00<?, ?it/s]


albumentations: Blur(p=0.01, blur_limit=(3, 7)), MedianBlur(p=0.01, blur_limit=(3, 7)), ToGray(p=0.01, method='weighted_average', num_output_channels=3), CLAHE(p=0.01, clip_limit=(1.0, 4.0), tile_grid_size=(8, 8))


/opt/anaconda3/lib/python3.12/site-packages/ultralytics/data/augment.py:1850: UserWarning: Argument(s) 'quality_lower' are not valid for transform ImageCompression
  A.ImageCompression(quality_lower=75, p=0.0),
/opt/anaconda3/lib/python3.12/site-packages/torch/utils/data/dataloader.py:692: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  warnings.warn(warn_msg)
val: Scanning /Users/hariraj/Desktop/STATBRIO/S_pdetection/Citypersons/labels/val/frankfurt.cache... 441 images, 61 backgrounds, 0 corrupt: 100%|██████████| 502/502 [00:00<?, ?it/s]

Plotting labels to runs_citypersons/yolov11_bifpn_pedestrian/labels.jpg... 


...

                   all        502       4164      0.195      0.162     0.0951     0.0324



2 epochs completed in 0.606 hours.
Optimizer stripped from runs_citypersons/yolov11_bifpn_pedestrian/weights/last.pt, 5.4MB
Optimizer stripped from runs_citypersons/yolov11_bifpn_pedestrian/weights/best.pt, 5.4MB

Validating runs_citypersons/yolov11_bifpn_pedestrian/weights/best.pt...
WARNING ⚠️ validating an untrained model YAML will result in 0 mAP.
Ultralytics 8.3.24 🚀 Python-3.12.7 torch-2.9.0 CPU (Apple M1)
YOLOv11_bifpn summary (fused): 238 layers, 2,582,347 parameters, 0 gradients, 6.3 GFLOPs


                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 32/32 [01:35<00:00,  3.00s/it]


                   all        502       4164      0.194      0.162     0.0949     0.0324
Speed: 0.4ms preprocess, 134.3ms inference, 0.0ms loss, 1.9ms postprocess per image
Results saved to runs_citypersons/yolov11_bifpn_pedestrian
MLflow: results logged to runs/mlflow
MLflow: disable with 'yolo settings mlflow=False'


ultralytics.utils.metrics.DetMetrics object with attributes:

ap_class_index: array([0])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x311674bf0>
curves: ['Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(B)']
curves_results: [[array([          0,    0.001001,    0.002002,    0.003003,    0.004004,    0.005005,    0.006006,    0.007007,    0.008008,    0.009009,     0.01001,    0.011011,    0.012012,    0.013013,    0.014014,    0.015015,    0.016016,    0.017017,    0.018018,    0.019019,     0.02002,    0.021021,    0.022022,    0.023023,
          0.024024,    0.025025,    0.026026,    0.027027,    0.028028,    0.029029,     0.03003,    0.031031,    0.032032,    0.033033,    0.034034,    0.035035,    0.036036,    0.037037,    0.038038,    0.039039,     0.04004,    0.041041,    0.042042,    0.043043,    0.044044,    0.045045,    0.046046,    0.047047,
          0.048048,    

In [11]:
metrics = model.val(data="citypersons.yaml")

print(f"mAP@0.5        : {metrics.box.map50:.4f}")
print(f"mAP@0.5:0.95   : {metrics.box.map:.4f}")
print(f"Precision      : {metrics.box.mp:.4f}")
print(f"Recall         : {metrics.box.mr:.4f}")

WARNING ⚠️ validating an untrained model YAML will result in 0 mAP.
Ultralytics 8.3.24 🚀 Python-3.12.7 torch-2.9.0 CPU (Apple M1)
YOLOv11_bifpn summary (fused): 238 layers, 2,582,347 parameters, 0 gradients, 6.3 GFLOPs


val: Scanning /Users/hariraj/Desktop/STATBRIO/S_pdetection/Citypersons/labels/val/frankfurt.cache... 441 images, 61 backgrounds, 0 corrupt: 100%|██████████| 502/502 [00:00<?, ?it/s]
/opt/anaconda3/lib/python3.12/site-packages/torch/utils/data/dataloader.py:692: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  warnings.warn(warn_msg)
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 63/63 [00:42<00:00,  1.48it/s]


                   all        502       4164      0.194      0.162     0.0949     0.0324
Speed: 0.4ms preprocess, 33.6ms inference, 0.0ms loss, 1.7ms postprocess per image
Results saved to runs_citypersons/yolov11_bifpn_pedestrian2
mAP@0.5        : 0.0949
mAP@0.5:0.95   : 0.0324
Precision      : 0.1945
Recall         : 0.1619


In [12]:
metrics.confusion_matrix.plot()
plt.show()

In [13]:
BEST_MODEL = "runs_citypersons/yolov11_bifpn_pedestrian/weights/best.pt"
assert os.path.exists(BEST_MODEL), "Best model not found"

print("Best model saved at:", BEST_MODEL)

Best model saved at: runs_citypersons/yolov11_bifpn_pedestrian/weights/best.pt


In [ ]:
import matplotlib
matplotlib.use("TkAgg")  # or "MacOSX"
import os
import cv2
from ultralytics import YOLO
import matplotlib.pyplot as plt

BEST_MODEL = "runs_citypersons/yolov11_bifpn_pedestrian/weights/best.pt"
assert os.path.exists(BEST_MODEL), "Best model not found"

model = YOLO(BEST_MODEL)

img_path = 'Citypersons/images/test/berlin/berlin_000038_000019_leftImg8bit.png'
img = cv2.imread(img_path)

results = model(img, conf=0.4)
annotated = results[0].plot()

plt.figure(figsize=(8, 8))
plt.imshow(cv2.cvtColor(annotated, cv2.COLOR_BGR2RGB))
plt.axis("off")
plt.show(block=False)
plt.pause(2)    
plt.close()   


0: 320x640 5 persons, 41.4ms
Speed: 2.2ms preprocess, 41.4ms inference, 3.8ms postprocess per image at shape (1, 3, 320, 640)


: 